## Домашнее задание 10: Уменьшение размеров модели

В этом задании вам предстоит научиться уменьшать размер модели, теряя при этом как можно меньше качества. Для этого вам предлагается реализовать две техники:
1. Факторизация матрицы эмбеддингов
1. Дистилляция

Вы будете решать задачу классификации токенов последовательности, а именно Named Entity Recognition (NER), на уже знакомом вам академическом датасете [CoNLL-2003](https://paperswithcode.com/dataset/conll-2003). В вашем распоряжении будет дообученный на эту задачу BERT, имеющий 100М весов. Вам необходимо будет сжать его до 20М.

### Напоминание о датасете

В CoNLL-2003 для именования сущностей используется маркировка **BIO** (Beggining, Inside, Outside), в которой метки означают следующее:

- *B-{метка}* – начало сущности *{метка}*
- *I-{метка}* – продолжнение сущности *{метка}*
- *O* – не сущность

Существуют так же и другие способы маркировки, например, BILUO. Почитать о них можно [тут](https://en.wikipedia.org/wiki/Inside–outside–beginning_(tagging)).

Всего в датасете есть 4 сущности: PER, ORG, LOC и MISC. Соответственно, 9 разных меток.
- O – слову не соответствует ни одна сущность.
- B-PER/I-PER – слово или набор слов соответстует определенному _человеку_.
- B-ORG/I-ORG – слово или набор слов соответстует определенной _организации_.
- B-LOC/I-LOC – слово или набор слов соответстует определенной _локации_.
- B-MISC/I-MISC – слово или набор слов соответстует сущности, которая не относится ни к одной из предыдущих. Например, национальность, произведение искусства, мероприятие и т.д.


Датасет лежит в папке `conll2003` и загрузить его можно таким образом.

In [2]:
from datasets import load_from_disk

dataset = load_from_disk("conll2003")

dataset

DatasetDict({
    train: Dataset({
        features: ['words', 'tags'],
        num_rows: 14041
    })
    test: Dataset({
        features: ['words', 'tags'],
        num_rows: 3453
    })
})

In [3]:
dataset['train'][0]

{'words': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}

In [4]:
label_names = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

In [5]:
words = dataset["train"][0]["words"]
labels = dataset["train"][0]["tags"]

for i in range(len(words)):
    print(f'{words[i]}\t{label_names[labels[i]]}')

EU	B-ORG
rejects	O
German	B-MISC
call	O
to	O
boycott	O
British	B-MISC
lamb	O
.	O


### Предобработка

На протяжении всего домашнего задания мы будем использовать _cased_ версию BERT, то есть токенизатор будет учитывать регистр слов. Для задачи NER регистр важен, так как имена и названия организаций или предметов искусства часто пишутся с большой буквы, и будет глупо прятать от модели такую информацию.

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

При токенизации слова могут разделиться на несколько токенов (как слово `lamb` из примера ниже), из-за чего появится несоответствие между числом токенов и тэгов. Это несоответствие нам придется устранить вручную.

In [7]:
words = dataset["train"][0]["words"]
inputs = tokenizer(words, is_split_into_words=True)

print('Слова: ', words)
print('Токены:', inputs.tokens())

Слова:  ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
Токены: ['[CLS]', 'EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'la', '##mb', '.', '[SEP]']


Функция `align_labels_with_tokens` выравнивает токены и их метки между собой.

In [9]:
import numpy as np


def align_labels_with_tokens(labels, word_ids):
    new_labels = np.full(len(word_ids), -100)  # -100 – специальное значение
    current_word = None
    for i, word_id in enumerate(word_ids[1:-1]):
        # следующее слово
        if word_id != current_word:
            current_word = word_id
            new_labels[i+1] = labels[word_id]
        # то же слово
        else:
            label = labels[word_id]
            if label % 2 == 1:
                label += 1  # меняем B- на I-
            new_labels[i+1] = label

    return new_labels

In [10]:
def tokenize_and_align(batch):
    # Токенизация и выравнивание батча текстов
    tokenized = tokenizer(batch["words"], is_split_into_words=True)
    all_labels = batch["tags"]
    aligned_labels = []
    for i, labels in enumerate(all_labels):
        aligned = align_labels_with_tokens(labels, tokenized.word_ids(i))
        aligned_labels.append(aligned)

    return {
        'input_ids': tokenized['input_ids'],
        'labels': aligned_labels
    }

Токенизируем все тексты, а так же выравниваем токены с метки.

In [11]:
tokenized_datasets = dataset.map(
    tokenize_and_align,
    batched=True,
    remove_columns=dataset['train'].column_names,
)

### Метрика

Для оценки качества NER обычно используют F1 меру. Мы загрузим ее из библиотеки `seqeval`.

In [12]:
# ! pip install seqeval

In [13]:
from seqeval.metrics import f1_score

Особенность подсчета F1 для NER заключается в том, что в некоторых ситуациях неправильные ответы могут засчитываться как правильные. Например, если модель предсказала `['I-PER', 'I-PER']`, то мы можем догадаться, что на самом деле должно быть `['B-PER', 'I-PER']`, так как сущность не может начинаться с `I-`. Функция `f1_score` учитывает это и поэтому работает только с текстовыми представлениями меток.

Мы будем считать F1 для набора текстов с помощью функции `compute_f1`. Она принимает пару (логиты модели, правильные ответы), очищает все от специальных токенов, преобразует метки в их текстовые имена и считает F1. Такой формат функций используется в `Trainer`.

In [15]:
def compute_f1(data):
    logits, labels = data
    predictions = np.argmax(logits, axis=-1)

    # Удаляем специальные токены и преобразуем в текстовые метки
    text_labels = []
    text_predictions = []
    for i in range(len(labels)):
        named_labels = []
        named_preds = []
        for j in range(labels.shape[1]):
            if labels[i, j] != -100:
                named_labels.append(label_names[labels[i, j]])
                named_preds.append(label_names[predictions[i, j]])

        text_labels.append(named_labels)
        text_predictions.append(named_preds)

    return {'f1': f1_score(text_labels, text_predictions)}

### Модель

В качестве начальной модели вам предоставляется дообученный BERT на задачу NER. Он хранится в файле `bert-base-cased-ner.pt` и получает примерно 0.9 F1 на тестовой выборке CoNLL-2003.

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [75]:
from transformers import AutoModelForTokenClassification, AutoConfig


config = AutoConfig.from_pretrained('bert-base-cased', num_labels=len(label_names))
model = AutoModelForTokenClassification.from_config(config).to(device)

model.load_state_dict(torch.load('bert-base-cased-ner.pt'))

print('Число параметров:', sum(p.numel() for p in model.parameters()))

Число параметров: 107726601


### Факторизация матрицы эмбеддингов

Можно заметить, что на данный момент матрица эмбеддингов занимает $V \cdot H = 28996 \cdot 768 = 22.268.928$ параметров. Это aж пятая часть от всей модели! Давайте попробуем что-то с этим сделать. В модели [ALBERT](https://arxiv.org/pdf/1909.11942.pdf) предлагается факторизовать матрицу эмбеддингов в произведение двух небольших матриц. Таким образом, параметры эмбеддингов будут содержать $V \cdot E + E \cdot H$ элементов, что гораздо меньше, если $H \gg E$. Авторы выбирают $E = 128$, однако ничего не мешает вам взять значение меньше.

__Задание 1.__ Допишите класс-обертку `FactorizedEmbedding`, который в конструкторе принимает на вход слой эмбеддингов `embedding` и пониженный ранг матрицы `rank`. Создайте новый слой эмбеддингов, реализующий произведение двух матриц. Выберите значение `rank` равным 64. Заметьте, обе матрицы можно инициализировать с помощью [SVD](https://pytorch.org/docs/stable/generated/torch.pca_lowrank.html) разложения, чтобы начальное приближение было хорошим. Это сэкономит очень много времени на дообучении.

In [ ]:
from torch import nn

class FactorizedEmbedding(nn.Module):
    def __init__(self, embedding: nn.Embedding, rank: int = 64):
        super().__init__()

        # ваш код здесь

    def forward(self, input_ids):
        # ваш код здесь
        pass

__Задание 2.__ Замените слой эмбеддингов на `FactorizedEmbedding` и дообучите модель с помощью `Trainer`. Качество после дообучения должно превысить 0.87 на тестовой выборке. Это не должно занять много времени. Для добавления паддингов к последовательностям используйте `DataCollatorForTokenClassification`.

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding=True, label_pad_token_id=-100)

In [ ]:
# ваш код здесь

Таким образом, нам удалось сэкономить __20М__ параметров и потерять в качестве всего пару процентов!

Для сдачи решения получите предсказания модели для датасета `grader_conll2003` и запишите их в файл `predictions.txt` в следующем формате.
```
O O O O B-LOC I-LOC I-LOC O O O O B-PEP O
O O B-PER I-PER O B-LOC I-LOC O O O O
O O B-PER I-PER I-PER O O O B-LOC O O
```

Каждая строка содержит предсказания для одного примера, записанные через пробел. Вы можете воспользоваться функцией `predict_and_save` для того, чтобы записать предсказания в нужном виде.

In [167]:
grader_dataset = load_from_disk("grader_conll2003")

tokenized_grader_dataset = grader_dataset.map(
    lambda sample: tokenizer(sample["words"], is_split_into_words=True),
    remove_columns=grader_dataset.column_names,
)

Map: 100%|██████████| 1000/1000 [00:00<00:00, 2821.74 examples/s]


In [77]:
@torch.inference_mode()
def predict_and_save(model, dataset, file_path='predictions.txt'):
    model.eval()
    with open(file_path, 'w') as f:
        for sample in dataset:
            logits = model(
                torch.tensor([sample['input_ids']], device=device)
            ).logits.cpu().squeeze(0)

            tags = logits.argmax(axis=-1)[1:-1]
            tag_names = [label_names[tag] for tag in tags]

            f.write(' '.join(tag_names))
            f.write('\n')

In [ ]:
# ваш код здесь

### Дистилляция знаний

Дистилляция знаний – это парадигма обучения, в которой знания модели-учителя дистиллируются в модель-ученика. Учеником может быть произвольная модель меньшего размера, решающая ту же задачу. При дистилляции используются два функционала ошибки:

1. Стандартная кросс-энтропия.
1. Функция, задающая расстояние между распределениями предсказаний учителя и ученика. Чаще всего используют KL-дивергенцию.

При этом для того, чтобы распределение предсказаний учителя не было таким вырожденным, к softmax добавляют температуру больше 1, например, 2 или 5.   
__Важно:__ при делении на температуру значения градиентов уменьшаются в $\tau^2$ раз. Поэтому для возвращения их в изначальный масштаб ошибку надо домножить на $\tau^2$. Подробнее об этом можно почитать в разделе 2.1 [оригинальной статьи](https://arxiv.org/pdf/1503.02531).

<img src="https://i.ibb.co/qsnyZZF/distillation.png" width="800"/>

__Задание 3.__ Реализуйте метод дистилляции знаний, изображенный на картинке. Для подсчета ошибки между предсказаниями ученика и учителя используйте KL-дивергенцию [`nn.KLDivLoss(reduction="batchmean")`](https://pytorch.org/docs/stable/generated/torch.nn.KLDivLoss.html) (обратите внимание на вормат ее входов). Для получения итоговой ошибки суммируйте мягкую ошибку с жесткой.   
В качестве учителя используйте дообученный BERT на задачу NER. В качестве ученика возьмите __произвольную__ необученную модель с размером __не больше 20M__ параметров. Вы можете использовать факторизацию матрицы эмбеддингов для уменьшения числа параметров. Если вы все сделали правильно, то на тестовой выборке вы должны получить значение F1 не меньше 0.73. Вам должно хватить примерно 20к итераций для этого.

__Важно:__ 
* Не забывайте добавлять _warmup_ при обучении ученика.
* Не забывайте переводить учителя в режим _eval_.

Для сдачи решения в напишите класс `Model`, который реализует вашу модель. Вы должны будете сдать этот класс вместе с весами модели `model.pt`. Мы загрузим вашу модель с помощью этого кода и замеряем качество на скрытой выборке.
```
model = Model()
model.load_state_dict(torch.load('model.pt'))
```

Решение не пройдет, если модель будет содержать больше 20М параметров.

In [ ]:
# ваш код здесь

### Резюме

Дистилляция – наиболее популярный и самый рабочий способ уменьшения размера модели с минимальной потерей качества. Сегодня нам удалось сжать модель в 5 раз и потерять около 15% F1. Без дистилляции потери были бы больше. Этот зазор можно уменьшить с помощью дополнительных приемов. Например, [обучая](https://arxiv.org/pdf/1909.103510) ученика сначала MLM задаче, [выравнивая](https://www.researchgate.net/profile/Alexander-Hernandez-13/publication/375758425_Knowledge_Distillation_Scheme_for_Named_Entity_Recognition_Model_Based_on_BERT/links/655b08ccb1398a779da06dcb/Knowledge-Distillation-Scheme-for-Named-Entity-Recognition-Model-Based-on-BERT.pdf) промежуточные слои ученика и учителя, или [удаляя](https://arxiv.org/pdf/1905.09418) ненужные головы для экономии параметров. Если вам хочется поглубже погрузиться в тему, то вы можете попробовать все это самостоятельно.